In [128]:
import pandas as pd
import numpy as np
import random

# Set seed for reproducibility
np.random.seed(42)

# 1. Product Dimension (JSON)
products = {
    'product_id': [f'P{i:03}' for i in range(1, 11)],
    'category': ['Electronics', 'Electronics', 'Home', 'Home', 'Apparel', 'Apparel', 'Books', 'Books', 'Toys', 'Toys'],
    'unit_cost': [200.0, 150.0, 50.0, 45.0, 15.0, 12.0, 10.0, 8.0, 25.0, 20.0],
    'base_price': [299.99, 199.99, 89.99, 79.99, 29.99, 24.99, 19.99, 14.99, 39.99, 34.99]
}
df_products = pd.DataFrame(products)
df_products.to_json(r'G:\campusx ML algo code\Practice Code\product_dim.json', orient='records')

# 2. Customer Profiles (Parquet)
customers = {
    'cust_id': [f'C{i:03}' for i in range(1, 101)],
    'signup_date': pd.date_range(start='2022-01-01', periods=100, freq='D'),
    'email': [f'user_{i}@GMAIL.com ' for i in range(1, 101)], # Note the trailing space/caps
    'membership': np.random.choice(['Gold', 'Silver', 'Bronze'], 100)
}
df_customers = pd.DataFrame(customers)
df_customers.to_csv(r'G:\campusx ML algo code\Practice Code\customer_profiles.csv', index=False)

# 3. Sales Data (CSV) - The "Dirty" Dataset
n_sales = 1000
sales = {
    'transaction_id': range(10001, 10001 + n_sales),
    'date': np.random.choice(pd.date_range('2023-01-01', '2023-12-31'), n_sales),
    'cust_id': [f'C{random.randint(1, 110):03}' for _ in range(n_sales)], # Some IDs won't match
    'product_id': [f'P{random.randint(1, 10):03}' for _ in range(n_sales)],
    'quantity': np.random.randint(1, 6, n_sales)
}
df_sales = pd.DataFrame(sales)

# Inject messiness
df_sales.loc[::50, 'quantity'] = 500  # Outliers
df_sales.loc[::30, 'product_id'] = np.nan  # Missing values

df_sales.to_csv(r'G:\campusx ML algo code\Practice Code\sales_data.csv', index=False)

print("Files generated: sales_data.csv, product_dim.json, customer_profiles.csv")

Files generated: sales_data.csv, product_dim.json, customer_profiles.csv


In [129]:
df_sales = pd.read_csv(r'G:\campusx ML algo code\Practice Code\sales_data.csv')

In [130]:
df_sales

,transaction_id,date,cust_id,product_id,quantity
0,10001,2023-05-16,C036,NaN,500
1,10002,2023-03-04,C046,P003,4
2,10003,2023-05-19,C078,P003,5
3,10004,2023-03-22,C043,P001,5
4,10005,2023-06-12,C087,P006,5
...,...,...,...,...,...
995,10996,2023-04-28,C023,P007,4
996,10997,2023-10-21,C048,P007,1
997,10998,2023-06-08,C088,P007,2
998,10999,2023-10-30,C057,P002,1


In [131]:
df_sales[df_sales['product_id'].isna()].shape

(34, 5)

In [132]:
df_customers.head()

,cust_id,signup_date,email,membership
0,C001,2022-01-01,user_1@GMAIL.com,Bronze
1,C002,2022-01-02,user_2@GMAIL.com,Gold
2,C003,2022-01-03,user_3@GMAIL.com,Bronze
3,C004,2022-01-04,user_4@GMAIL.com,Bronze
4,C005,2022-01-05,user_5@GMAIL.com,Gold


In [133]:
df_products.head()

,product_id,category,unit_cost,base_price
0,P001,Electronics,200.0,299.99
1,P002,Electronics,150.0,199.99
2,P003,Home,50.0,89.99
3,P004,Home,45.0,79.99
4,P005,Apparel,15.0,29.99


### Phase 1: Data Acquisition & Structural Integrity
Concepts: IO Tools, Inspection, Memory Management

Task: Load the datasets. Convert the sales_data timestamps into proper datetime64 objects.

Optimization: Downcast numeric types (e.g., float64 to float32) to save memory.

Validation: Use .info(), .describe(), and .shape to check for data types and distribution.

In [134]:
df_sales.dtypes

transaction_id    int64
date                str
cust_id             str
product_id          str
quantity          int64
dtype: object

In [135]:
df_sales['cust_id'] = df_sales['cust_id'].astype('string')
df_sales['product_id'] = df_sales['product_id'].astype('string')

In [136]:
df_sales['transaction_id'] = pd.to_numeric(df_sales['transaction_id'], downcast='integer')

In [137]:
df_sales.dtypes

transaction_id     int16
date                 str
cust_id           string
product_id        string
quantity           int64
dtype: object

In [138]:
df_sales.describe()

,transaction_id,quantity
count,1000.000000,1000.000000
mean,10500.500000,12.938000
std,288.819436,69.629393
min,10001.000000,1.000000
25%,10250.750000,2.000000
50%,10500.500000,3.000000
75%,10750.250000,4.000000
max,11000.000000,500.000000


### Phase 2: The "Clean-up" Crew
Concepts: Handling Missing Data, String Manipulation, Vectorized Operations

Dealing with Nulls: Use .fillna() with business logic (e.g., fill missing prices with the category median) or .dropna() where records are unusable.

Regex Power: Standardize the Customer_Email column (lowercase, stripping whitespace) and extract "Country Codes" from a messy Phone_Number string column using .str.extract().

Outlier Detection: Identify "fat-finger" errors in quantity using Z-scores or IQR filtering.

In [139]:
df_sales.isnull().sum()

transaction_id     0
date               0
cust_id            0
product_id        34
quantity           0
dtype: int64

In [140]:
df_sales = df_sales.dropna()

In [141]:
df_sales.shape

(966, 5)

In [142]:
df_customers.isnull().sum()

cust_id        0
signup_date    0
email          0
membership     0
dtype: int64

In [143]:
df_products.isnull().sum()

product_id    0
category      0
unit_cost     0
base_price    0
dtype: int64

In [144]:
df_customers.head()

,cust_id,signup_date,email,membership
0,C001,2022-01-01,user_1@GMAIL.com,Bronze
1,C002,2022-01-02,user_2@GMAIL.com,Gold
2,C003,2022-01-03,user_3@GMAIL.com,Bronze
3,C004,2022-01-04,user_4@GMAIL.com,Bronze
4,C005,2022-01-05,user_5@GMAIL.com,Gold


### Action: Perform an inner join between Sales and Products, and a left join with Customers.

In [145]:
df_merger = pd.merge(df_sales,df_products, on='product_id', how='inner')

In [146]:
final_merger = pd.merge(df_merger,df_customers, on='cust_id', how='left')

In [147]:
final_merger.shape

(966, 11)

In [148]:
final_merger['cust_id'].nunique()

110

### Email Cleanup: Convert the email column to lowercase and remove leading/trailing spaces using .str methods.

In [149]:
final_merger.head(3)

,transaction_id,date,cust_id,product_id,quantity,category,unit_cost,base_price,signup_date,email,membership
0,10002,2023-03-04,C046,P003,4,Home,50.0,89.99,2022-02-15,user_46@GMAIL.com,Bronze
1,10003,2023-05-19,C078,P003,5,Home,50.0,89.99,2022-03-19,user_78@GMAIL.com,Bronze
2,10004,2023-03-22,C043,P001,5,Electronics,200.0,299.99,2022-02-12,user_43@GMAIL.com,Bronze


In [150]:
final_merger[final_merger['email'].str.len()==17]

,transaction_id,date,cust_id,product_id,quantity,category,unit_cost,base_price,signup_date,email,membership
4,10006,2023-10-16,C006,P006,4,Apparel,12.0,24.99,2022-01-06,user_6@GMAIL.com,Gold
7,10009,2023-02-10,C006,P004,2,Home,45.0,79.99,2022-01-06,user_6@GMAIL.com,Gold
16,10018,2023-08-04,C006,P009,5,Toys,25.0,39.99,2022-01-06,user_6@GMAIL.com,Gold
18,10020,2023-04-09,C005,P004,3,Home,45.0,79.99,2022-01-05,user_5@GMAIL.com,Gold
39,10042,2023-08-29,C005,P005,1,Apparel,15.0,29.99,2022-01-05,user_5@GMAIL.com,Gold
...,...,...,...,...,...,...,...,...,...,...,...
935,10969,2023-11-05,C009,P008,2,Books,8.0,14.99,2022-01-09,user_9@GMAIL.com,Bronze
936,10970,2023-04-14,C003,P007,4,Books,10.0,19.99,2022-01-03,user_3@GMAIL.com,Bronze
939,10973,2023-07-17,C009,P005,1,Apparel,15.0,29.99,2022-01-09,user_9@GMAIL.com,Bronze
940,10974,2023-05-14,C003,P008,5,Books,8.0,14.99,2022-01-03,user_3@GMAIL.com,Bronze


In [151]:
leading = final_merger['email'].str.startswith(' ')
trailing = final_merger['email'].str.endswith(' ')
has_either = leading | trailing

In [152]:
final_merger[final_merger['email'].str.endswith(' ')]

,transaction_id,date,cust_id,product_id,quantity,category,unit_cost,base_price,signup_date,email,membership
0,10002,2023-03-04,C046,P003,4,Home,50.0,89.99,2022-02-15,user_46@GMAIL.com,Bronze
1,10003,2023-05-19,C078,P003,5,Home,50.0,89.99,2022-03-19,user_78@GMAIL.com,Bronze
2,10004,2023-03-22,C043,P001,5,Electronics,200.0,299.99,2022-02-12,user_43@GMAIL.com,Bronze
3,10005,2023-06-12,C087,P006,5,Apparel,12.0,24.99,2022-03-28,user_87@GMAIL.com,Bronze
4,10006,2023-10-16,C006,P006,4,Apparel,12.0,24.99,2022-01-06,user_6@GMAIL.com,Gold
...,...,...,...,...,...,...,...,...,...,...,...
960,10995,2023-12-19,C023,P004,5,Home,45.0,79.99,2022-01-23,user_23@GMAIL.com,Silver
961,10996,2023-04-28,C023,P007,4,Books,10.0,19.99,2022-01-23,user_23@GMAIL.com,Silver
962,10997,2023-10-21,C048,P007,1,Books,10.0,19.99,2022-02-17,user_48@GMAIL.com,Gold
963,10998,2023-06-08,C088,P007,2,Books,10.0,19.99,2022-03-29,user_88@GMAIL.com,Gold


In [153]:
final_merger.head(2)

,transaction_id,date,cust_id,product_id,quantity,category,unit_cost,base_price,signup_date,email,membership
0,10002,2023-03-04,C046,P003,4,Home,50.0,89.99,2022-02-15,user_46@GMAIL.com,Bronze
1,10003,2023-05-19,C078,P003,5,Home,50.0,89.99,2022-03-19,user_78@GMAIL.com,Bronze


In [154]:
final_merger['email'] = final_merger['email'].str.lower()

In [155]:
final_merger['email'] = final_merger['email'].str.strip()
final_merger['email'].str.len().value_counts()

email
17.0    795
16.0     95
18.0      3
Name: count, dtype: int64

### Outlier Handling: The quantity column has values of 500 (errors). Replace any quantity > 10 with the median quantity of that specific product category.

In [156]:
final_merger[final_merger['quantity']>10]

,transaction_id,date,cust_id,product_id,quantity,category,unit_cost,base_price,signup_date,email,membership
48,10051,2023-02-05,C056,P007,500,Books,10.0,19.99,2022-02-25,user_56@gmail.com,Bronze
96,10101,2023-04-23,C028,P005,500,Apparel,15.0,29.99,2022-01-28,user_28@gmail.com,Bronze
193,10201,2023-10-04,C106,P006,500,Apparel,12.0,24.99,NaT,NaN,NaN
241,10251,2023-11-19,C085,P003,500,Home,50.0,89.99,2022-03-26,user_85@gmail.com,Gold
338,10351,2023-05-06,C038,P006,500,Apparel,12.0,24.99,2022-02-07,user_38@gmail.com,Bronze
386,10401,2023-09-04,C065,P008,500,Books,8.0,14.99,2022-03-06,user_65@gmail.com,Silver
483,10501,2023-03-17,C088,P005,500,Apparel,15.0,29.99,2022-03-29,user_88@gmail.com,Gold
531,10551,2023-07-09,C012,P003,500,Home,50.0,89.99,2022-01-12,user_12@gmail.com,Bronze
628,10651,2023-07-21,C088,P007,500,Books,10.0,19.99,2022-03-29,user_88@gmail.com,Gold
676,10701,2023-12-01,C082,P005,500,Apparel,15.0,29.99,2022-03-23,user_82@gmail.com,Gold


In [157]:
median_val = final_merger['quantity'].median()

In [158]:
final_merger.loc[final_merger['quantity'] > 10, 'quantity'] = median_val

In [159]:
final_merger['quantity'].describe()

count    966.000000
mean       2.991718
std        1.409786
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        5.000000
Name: quantity, dtype: float64

In [160]:
final_merger.columns

Index(['transaction_id', 'date', 'cust_id', 'product_id', 'quantity',
       'category', 'unit_cost', 'base_price', 'signup_date', 'email',
       'membership'],
      dtype='str')

### Financial Analysis (Grouped)
Calculate Margin: Margin = (Base_Price - Unit_Cost) * Quantity.

Grouped Aggs: Find the Total Revenue and Average Margin per Category and Membership level.

In [161]:
final_merger['Margin'] = (final_merger['base_price'] - final_merger['unit_cost']) * final_merger['quantity']

In [162]:
final_merger.head(2)

,transaction_id,date,cust_id,product_id,quantity,category,unit_cost,base_price,signup_date,email,membership,Margin
0,10002,2023-03-04,C046,P003,4,Home,50.0,89.99,2022-02-15,user_46@gmail.com,Bronze,159.96
1,10003,2023-05-19,C078,P003,5,Home,50.0,89.99,2022-03-19,user_78@gmail.com,Bronze,199.95


In [163]:
final_merger.rename(columns={'Margin':'margin'},inplace=True)

In [164]:
final_merger.isnull().sum()

transaction_id     0
date               0
cust_id            0
product_id         0
quantity           0
category           0
unit_cost          0
base_price         0
signup_date       73
email             73
membership        73
margin             0
dtype: int64

In [169]:
final_merger.dropna(inplace=True)

In [170]:
final_merger.shape

(893, 12)

In [173]:
final_merger.groupby(by=['category'])['margin'].mean()

category
Apparel         40.251257
Books           26.089844
Electronics    225.103583
Home           112.952164
Toys            47.022202
Name: margin, dtype: float64

In [174]:
final_merger.groupby(by=['membership'])['margin'].mean()

membership
Bronze    92.628759
Gold      90.843436
Silver    89.949872
Name: margin, dtype: float64

In [175]:
# Revenue = How much the customer paid
final_merger['revenue'] = final_merger['quantity'] * final_merger['base_price']

In [178]:
final_merger.groupby(by=['category'])['revenue'].sum()

category
Apparel         13844.97
Books           10269.25
Electronics    140494.37
Home            43814.82
Toys            19684.73
Name: revenue, dtype: float64

In [179]:
final_merger.groupby(by=['membership'])['revenue'].sum()

membership
Bronze    75491.34
Gold      73246.44
Silver    79370.36
Name: revenue, dtype: float64

In [181]:
final_merger.groupby(by=['category','membership'])['revenue'].sum()

category     membership
Apparel      Bronze         4673.33
             Gold           4343.40
             Silver         4828.24
Books        Bronze         3503.06
             Gold           2918.42
             Silver         3847.77
Electronics  Bronze        47498.09
             Gold          43698.28
             Silver        49298.00
Home         Bronze        14508.31
             Gold          15538.14
             Silver        13768.37
Toys         Bronze         5308.55
             Gold           6748.20
             Silver         7627.98
Name: revenue, dtype: float64

In [182]:
final_merger.groupby(by=['category','membership'])['margin'].mean()

category     membership
Apparel      Bronze         41.567193
             Gold           39.578571
             Silver         39.649032
Books        Bronze         25.192059
             Gold           26.563333
             Silver         26.596714
Electronics  Bronze        225.366508
             Gold          224.547119
             Silver        225.353846
Home         Bronze        112.426491
             Gold          110.292581
             Silver        116.699423
Toys         Bronze         48.301111
             Gold           44.970000
             Silver         48.063175
Name: margin, dtype: float64

In [180]:
# We use .agg() to perform different math on different columns
analysis = final_merger.groupby(['category', 'membership']).agg(
    total_revenue=('revenue', 'sum'),
    avg_margin=('margin', 'mean'),
    transaction_count=('transaction_id', 'count')
).reset_index()

print(analysis)

       category membership  total_revenue  avg_margin  transaction_count
0       Apparel     Bronze        4673.33   41.567193                 57
1       Apparel       Gold        4343.40   39.578571                 56
2       Apparel     Silver        4828.24   39.649032                 62
3         Books     Bronze        3503.06   25.192059                 68
4         Books       Gold        2918.42   26.563333                 54
5         Books     Silver        3847.77   26.596714                 70
6   Electronics     Bronze       47498.09  225.366508                 63
7   Electronics       Gold       43698.28  224.547119                 59
8   Electronics     Silver       49298.00  225.353846                 65
9          Home     Bronze       14508.31  112.426491                 57
10         Home       Gold       15538.14  110.292581                 62
11         Home     Silver       13768.37  116.699423                 52
12         Toys     Bronze        5308.55   48.3011

### The Time-Series Boss
Growth: Resample the data to see Monthly Total Revenue.

Windowing: Calculate a 3-month rolling average of revenue to identify seasonal trends.

Peak Hours: If you add a "Time" component, find which hour of the day generates the most "Gold" member sales.

In [184]:
final_merger.dtypes

transaction_id             int16
date                         str
cust_id                   object
product_id                object
quantity                   int64
category                     str
unit_cost                float64
base_price               float64
signup_date       datetime64[us]
email                        str
membership                   str
margin                   float64
revenue                  float64
dtype: object

In [185]:
# Ensure date is datetime type
final_merger['date'] = pd.to_datetime(final_merger['date'])

# Set the index to the date column
df_time = final_merger.set_index('date').sort_index()

In [187]:
df_time.head(3)

,transaction_id,cust_id,product_id,quantity,category,unit_cost,base_price,signup_date,email,membership,margin,revenue
date,,,,,,,,,,,,
2023-01-01,10892,C023,P009,3,Toys,25.0,39.99,2022-01-23,user_23@gmail.com,Silver,44.97,119.97
2023-01-01,10549,C032,P007,3,Books,10.0,19.99,2022-02-01,user_32@gmail.com,Bronze,29.97,59.97
2023-01-01,10578,C022,P008,4,Books,8.0,14.99,2022-01-22,user_22@gmail.com,Gold,27.96,59.96


In [188]:
monthly_revenue = df_time['revenue'].resample('ME').sum()

monthly_revenue

date
2023-01-31    17727.94
2023-02-28    16062.77
2023-03-31    22082.90
2023-04-30    20502.42
2023-05-31    19047.61
2023-06-30    17847.62
2023-07-31    15643.03
2023-08-31    21982.44
2023-09-30    16827.96
2023-10-31    22102.66
2023-11-30    19792.93
2023-12-31    18487.86
Freq: ME, Name: revenue, dtype: float64

In [190]:
# Calculate the 3-month rolling average
# window=3 means it looks at the current month + 2 previous months
monthly_trends = monthly_revenue.rolling(window=3).mean()

# Combine for comparison
comparison = pd.DataFrame({
    'Actual_Revenue': monthly_revenue,
    '3_Month_Moving_Avg': monthly_trends
})

print(comparison)

            Actual_Revenue  3_Month_Moving_Avg
date                                          
2023-01-31        17727.94                 NaN
2023-02-28        16062.77                 NaN
2023-03-31        22082.90        18624.536667
2023-04-30        20502.42        19549.363333
2023-05-31        19047.61        20544.310000
2023-06-30        17847.62        19132.550000
2023-07-31        15643.03        17512.753333
2023-08-31        21982.44        18491.030000
2023-09-30        16827.96        18151.143333
2023-10-31        22102.66        20304.353333
2023-11-30        19792.93        19574.516667
2023-12-31        18487.86        20127.816667


In [193]:
final_merger['hour'] = np.random.randint(0, 24, size=len(final_merger))

In [200]:
final_merger.head(3)

,transaction_id,date,cust_id,product_id,quantity,category,unit_cost,base_price,signup_date,email,membership,margin,revenue,hour
0,10002,2023-03-04,C046,P003,4,Home,50.0,89.99,2022-02-15,user_46@gmail.com,Bronze,159.96,359.96,23
1,10003,2023-05-19,C078,P003,5,Home,50.0,89.99,2022-03-19,user_78@gmail.com,Bronze,199.95,449.95,9
2,10004,2023-03-22,C043,P001,5,Electronics,200.0,299.99,2022-02-12,user_43@gmail.com,Bronze,499.95,1499.95,7


In [204]:
gold_members = final_merger[final_merger['membership']=="Gold"].reset_index()

In [210]:
gold_members.groupby(['hour'])['revenue'].sum().sort_values(ascending=False).head(1)

hour
17    5414.46
Name: revenue, dtype: float64